In [ ]:
# -*- coding: utf-8 -*-

Tarea2_IA_JavieraManzo.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/12gEn8evbxCpT0tjrLr9BYjfxDOwL7bOj

# Tarea Nº 1 - Inteligencia Artificial
### Alumno: Javiera Manzo González
### Fecha de entrega: 29 de Octubre 2024

## Introducción
Para el desarrollo de la segunda tarea del ramo de inteligencia artificial, se abordarán temas de Machine learning tales como clasificación y clustering aplicadas a un dataset sobre el spam en correos electrónicos. El objetivo es observar y comparar los resultados de diferentes modelos de aprendizaje supervisado y no supervisado con el fin de determinar patrones en los correos.
El primer ítem se enfoca en entrenar un modelo de Regresión Logística para clasificar los correos que corresponden a spam o no, utilizando métricas de error como la matriz de confusión y la precisión para evaluar el rendimiento. Para el ítem siguiente se buscan generar las mismas métricas anteriores, pero esta vez con el algoritmo K- Nearest Neighbors (KNN) con diversos valores de K con el fin de observar como afecta este cambio los resultados. Para finalizar se aplican los métodos de clustering DBSCAN y GMM los cuales permiten observar estructuras de agrupamiento en los datos sin la necesidad de etiquetas explícitas.
Esta actividad no solo tiene como propósito construir modelos precisos, sino también comprender el funcionamiento de estos algoritmos, aplicando de forma práctica los conceptos trabajados teóricamente en clase. Así, se busca proporcionar una visión integral sobre el análisis de datos y el reconocimiento de patrones en el contexto de la detección de spam mediante técnicas de Machine Learning.

Se establecen las librerias con las que se va a trabajar si bien no son todas, esta es la lista que se consideran necesaria en una primera instancia.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.cluster import DBSCAN
from sklearn.mixture import GaussianMixture

A continuación se desarrollará una lista con una breve explicación de cada una y el rol que cumplen en el contexto de la tarea:


1.  **NumPy** corresponde a una biblioteca de soporte para operaciones numéricas en Python. Esta nos ofrece herramientas para trabajar con grandes matrices y realizar operaciones matemáticas complejas.
2.  **panda** es una biblioteca que nos permite la manipulación y análisis de datos estructurados. Esta nos permite leer, organizar y procesar los datos de manera eficiente.
3. ** train_test_split** función de **sklearn.model_selection** usada para dividir el dataset en conjuntos de entrenamiento y prueba de forma aleatoria.
4. **LogisticRegression** función de **sklearn.linear_model** que corresponde al algoritmo de clasificación supervisado basado en regresión logística.
5. **KNeighborsClassifier** función de **sklearn.neighbors** que nos permite implementar el algoritmo K-Nearest Neighbors (KNN).
6. **accuracy_score, confusion_matrix** de **sklearn.metrics** que corresponden a métricas de evaluación de modelos.
7. **DBSCAN** una función de **sklearn.cluster** la cual nos permite implementar el algoritmo de clustering no supervisado Density-Based Spatial Clustering of Applications with Noise (DBSCAN).
8. **GaussianMixture** función de **sklearn.mixture** la cual nos permite la implementación del modelo clustering de mezclas Gaussianas (GMM).

##ataset
Como siguiente paso se establece el entorno y la obtención del dataset. En esta oportunidad se utilizó uno llamado “Spam Email Classification Dataset”, el cual corresponde al registro de 83446 correos electrónicos los cuales están clasificados en sí son spam o no, este archivo solo cuenta con 2 columnas (Label, Text), las cuales son el contenido del correo electrónico y la etiqueta binaria siendo “1” el indicador de que el correo está clasificado como spam y “0” cuando no.

In [ ]:
from google.colab import files
uploaded = files.upload()

!pip install -q kaggle

! mkdir ~/.kaggle
! cp kaggle.json ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d purusinghvi/email-spam-classification-dataset

!unzip /content/email-spam-classification-dataset.zip #extraccion del archivo .csv

data= pd.read_csv('combined_data.csv')


print(data.shape)
data.head()

##Preparación de los datos
Ya que el dataset originalmente es solo un archivo y nos piden en el enunciado trabajar con conjuntos DISJUNTOS se utiliza la función antes mencionada **train_test_split**, primero se establece los parámetros correspondientes a X e Y los cuales son la columna “text” en caso de X y la etiqueta binaria “label” en el caso de Y. Con esos datos establecidos se utiliza la función que tiene como parámetros, X, Y, “test_size” que corresponde al porcentaje del archivo el cual se quiere dejar para “test” y “random_state” que corresponde a la semilla usada para la generación de números aleatorios.

In [ ]:
X = data.drop('label', axis=1)['text'].str.lower().str.replace(r'[^\w\s]', '')
y = data['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

La función establece 4 conjuntos de datos los cuales corresponden al X e Y de los datos usados para el entrenamiento (train) y los usados para el posterior testeo(test) dando como resultado "X_train", "y_train", "X_test" y "y_test"

In [ ]:
print('train: ',X_train.shape)
print('test: ',X_test.shape)

Entonces podemos ver cuantas filas tiene cada conjunto de datos siendo “train” quien tiene la mayoría con un 66% (55910) y “test” teniendo el resto que corresponde a un 33% (27538).

In [ ]:
y_train.value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['pink','slateblue'],title='Distribución de SPAM [TRAIN]')

Se utiliza la función “plot” para desarrollar el gráfico de torta que nos permite apreciar la distribución de SPAM presente en el conjunto “train”, si bien los porcentajes están bastante cercanos podemos apreciar que es un dataset desequilibrado.

In [ ]:
y_test.value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['pink','slateblue'],title='Distribución de SPAM [TEST]')

Lo mismo que en el gráfico anterior en el conjunto de “test” también se tiene un dataset desequilibrado presentando un porcentaje mayor en 1 lo que significa que más del 50% de los correos en ambos conjuntos pertenecen a la clasificación de SPAM.

##Regresión Logística

Antes de poder trabajar con los algoritmos deseados se deben codificar los datos, ya que dichos algoritmos solo permiten datos del tipo numéricos y el dataset elegido tiene texto. Con la ayuda de la función **TfidfVectorizer**(Term Frequency-Inverse Document Frequency), recomendada por la ayudante de la asignatura, esta función transforma cada palabra según su importancia y frecuencia en el documento.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

vectorizer_x = TfidfVectorizer()
vectorizer_y = LabelEncoder()

train_x = vectorizer_x.fit_transform(X_train)
train_y =  vectorizer_y.fit_transform(y_train)
test_x =  vectorizer_x.transform(X_test)
test_y =  vectorizer_y.transform(y_test)

A pesar de que la columna “label” ya se encuentra en datos binarios, se realiza la codificación a estos datos también.

In [ ]:
print(train_x.shape)
print(train_y.shape)
print(test_x.shape)
print(test_y.shape)

Una vez se tenga todo el dataset correctamente preparado para su utilización, se implementa la instancia de Regresión logística, gracias al parámetro “verbose=true” podemos observar mensajes de progreso durante el entrenamiento.
Una vez establecida la instancia se comienza el entreno con la funcion ".fit" la cual recibe los parámetros de X e Y del conjunto “train”.
El algoritmo de regresión logística corresponde a un método de clasificación supervisado que predice la probabilidad de pertenecer a una clase específica diseñada para que, a diferencia de la regresión lineal, se desarrollen con este problema de clasificación binaria.

In [ ]:
logistic_regression = LogisticRegression(verbose = True)
logistic_regression.fit(train_x, train_y)

Una vez acabado el entrenamiento se utiliza el modelo para hacer predicciones sobre los datos X del conjunto de pruebas, utilizando “predict” el cual genera las etiquetas predichas para cada entrada de X.

In [ ]:
y_pred_l = logistic_regression.predict(test_x)

##Métricas
Las métricas escogidas corresponden a la precisión y matriz de confusión. La primera mide el porcentaje de mediciones correctas entre el total de predicciones realizadas, esta función toma los datos obtenidos en la predicción realiza a los datos X del conjunto de entrenamiento y los compara con los datos reales Y que se encuentran en el conjunto de testeo.
La segunda métrica seleccionada es la matriz de confusión, esta es una tabla de 2x2 que resume las predicciones del modelo, en dicho cuadro podemos cotejar los datos reales con las predicciones realizadas, teniendo en la esquina superior izquierda los datos reales positivos (correo que si es spam) predicho correctamente, en la esquina superior derecha los datos reales positivos predichos erróneamente, la esquina inferior izquierda los datos reales negativos (correos que no son spam) predichos erróneamente como spam y por último en la esquina inferior derecha los datos reales negativos predichos como no spam.

In [ ]:
accuracy_log = accuracy_score(test_y, y_pred_l)
conf_matrix_log = confusion_matrix(test_y, y_pred_l)

print("Métricas de Regresión Logística:")
print("Accuracy:", accuracy_log)
print("Matriz de Confusión:\n", conf_matrix_log)

##KNN
Para la segunda parte de esta experiencia se nos solicita utilizar el algoritmo de clasificación supervisado K-Nearest Neighbors (KNN). Dicho algoritmo consiste en asignar una clase a una muestra en función a las clases de sus “K” vecinos más cercanos en el espacio. Primero se establece un número entero “K” el cual representa la cantidad de vecinos a considerar, posteriormente se establece la distancia(la más común a utilizar es la Euclidiana) entre la nueva muestra y todas las muestras del conjunto de datos, una vez realizados todos los cálculos se identifican los K vecinos más cercanos y se establece la clase según sea la mayoritaria entre sus vecinos.

En este caso se aplica el algoritmo 5 veces con distintos números de K (1,3,5,7,9).

In [ ]:
# Prueba diferentes valores de K
k_values = [1,3,5,7,9]
accuracy_knn = {}
conf_matrix_knn = {}

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(train_x, train_y)
    y_pred_knn = knn.predict(test_x)

    # Calcular y almacenar métricas
    accuracy_knn[k] = accuracy_score(test_y, y_pred_knn)
    conf_matrix_knn[k] = confusion_matrix(test_y, y_pred_knn)

print("Resultados KNN:")
for k, acc in accuracy_knn.items():
    print(f"K={k} - Accuracy: {acc}")
    print(f"Matriz de Confusión para K={k}:\n{conf_matrix_knn[k]}")

Para el desarrollo del código se establecieron los valores de K con los que se  realizó la ejecución del algoritmo, luego se inicializaron 2 arreglos vacíos, el primero en el que se guardan los datos de la métrica de exactitud para cada k y el otro donde se guarda la matriz de confusión de cada uno de los mismos.
Luego dentro de un ciclo “for” el cual se corre en todos los valores establecidos de K, se utiliza la función **KNeighborsClassifier** lo que nos permite utilizar el algoritmo, este recibe el valor de K, luego se realiza el entrenamiento con “fit” el cual recibe al igual que en el algoritmo utilizado anteriormente los valores de X e Y del conjunto “train”, luego del entrenamiento se realizan las predicciones y se guardan los datos obtenidos en los arreglos antes mencionados.
Por último con un conjunto de prints podemos apreciar todos los valores almacenados en los arreglos que corresponden a la exactitud del algoritmo implementado con los distintos K y sus respectivas matrices de confusión.

##Analisis comparativo item 1 y 2

In [ ]:
print("Métricas de Regresión Logística:")
print("Accuracy:", accuracy_log)
print("Matriz de Confusión:\n", conf_matrix_log)

Gracias a los datos obtenidos en el primer ítem podemos notar que el porcentaje de exactitud obtenido con la regresión logística es de un 98.44%, un accuracy dándonos da a entender que el modelo hace buenas predicciones y tiene un alto rendimiento. Además, gracias a la matriz de confusión podemos darnos cuenta de que los errores cometidos como clasificar correos spam como no spam y viceversa no supera los 500 errores(310 y 119), dentro de un conjunto de más de 27.000 datos es un número muy bajo representado por menos de 4% del total.

In [ ]:
print("Resultados KNN:")
for k, acc in accuracy_knn.items():
    print(f"K={k} - Accuracy: {acc}")
    print(f"Matriz de Confusión para K={k}:\n{conf_matrix_knn[k]}")

Por otro lado, observando las métricas obtenidas con el algoritmo KNN con K igual a 1, 3, 5, 7 y 9 podemos darnos cuenta de que a medida que aumenta el tamaño de K la exactitud baja, lo que sugiere que la regresión logística es más confiable que KNN en este tipo de tareas de clasificación.
Por otro lado, de la matriz de confusión podemos darnos cuenta de que KNN(48, 32, 25, 31 y 44) tiene valores más bajos que la regresión logística(119) para los falsos negativos(esquina inferior izquierda), esto puede significar que los segundos cometen más errores a la hora de clasificar correos que no son spam como spam, por el lado contrario KNN excede mucho más sus valores en comparación a la regresión logística cuando se trata de falsos positivos, teniendo números tales como 3289,5011, 5784,6547 y 7056 en comparación a los 310 errores, esto quiere decir que KNN deja pasar muchos correos que sí son spam como no spam.

##DBSCAN y GMM
Para la ultima parte de esta experiencia se nos pide utilizar el algoritmo de clustering basado en la densidad llamado Density-Based Spatial Clustering of Applications with Noise(DBSCAN), Este algoritmo es util cuando se trata de identificar grupos de datos en conjutos donde las densidades son variables, en el contexto del dataset utilizado este algoritmo nos permite identificar las areas mas densas que se traducen en las palabras mas utilizadas en los correos electronicos registrados.
Este algoritmo cuenta con conceptos basicos los cuales corresponden a los parametros a establecer como "eps" el cual es la distancia maxima entre dos puntos para que uno se considere vecino del otro, ademas tenemos el valor de "min_samples" lo que nos determina el numero minimo de puntos necesarios para que un conjunto se considere un "nucleo".
El funcionamiento de este algoritmo

In [ ]:
# DBSCAN con distintas configuraciones
dbscan_configs = [(0.3, 5), (0.5, 5), (0.3, 10), (0.5, 10)] # (eps, min_samples)
dbscan_labels = {}

for eps, min_samples in dbscan_configs:
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(train_x)
    dbscan_labels[(eps, min_samples)] = labels
    print(f"DBSCAN con eps={eps}, min_samples={min_samples} - Etiquetas encontradas: {np.unique(labels)}")

Estos datos denotan los clusters encontrados para cada configuración establecida, siendo 136 los encontrados para la primera configuración, 338 para la segunda, 53 para la tercera y 147 para la última, también podemos apreciar que en todas se presenta ruido, esto visible gracias al -1 que se encuentra en el principio de todas.